# Fixed-Length Token Window Chunking


1. **No overlap** (0%) -> `chunks_fixed300_overlap0.jsonl`
2. **With overlap** (20%) -> `chunks_fixed300_overlap20pct.jsonl`

In [ ]:
%pip install tiktoken

In [1]:
import json
import os
import tiktoken
from typing import List, Dict, Any
from collections import Counter

In [4]:
# Config
CORPUS_PATHS = [
    "../data/corpus_wikipedia.jsonl",
    "../data/corpus_wikibooks_east_asia.jsonl",
    "../data/corpus_blog.jsonl",
]

CHUNK_SIZE = 300  # tokens per chunk

ENCODING = tiktoken.get_encoding("cl100k_base")

## Load Corpus

In [5]:
def load_corpus(paths: List[str]) -> List[Dict[str, Any]]:
    documents = []
    for path in paths:
        if not os.path.exists(path):
            print(f"[WARN] File not found, skipped: {path}")
            continue
        count = 0
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    documents.append(json.loads(line))
                    count += 1
        print(f"Loaded {count} docs from {path}")
    print(f"Total: {len(documents)} documents")
    return documents

documents = load_corpus(CORPUS_PATHS)

Loaded 223 docs from ../data/corpus_wikipedia.jsonl
Loaded 78 docs from ../data/corpus_wikibooks_east_asia.jsonl
Loaded 30 docs from ../data/corpus_blog.jsonl
Total: 331 documents


## Chunking Functions

In [6]:
def fixed_length_token_chunk(
    text: str,
    chunk_size: int = 300,
    overlap_pct: float = 0.0,
    encoding: tiktoken.Encoding = ENCODING,
) -> List[str]:
    
    if not text or not text.strip():
        return []

    tokens = encoding.encode(text)
    total = len(tokens)

    if total <= chunk_size:
        return [text.strip()]

    overlap = int(chunk_size * overlap_pct)
    step = chunk_size - overlap
    chunks = []

    for start in range(0, total, step):
        end = min(start + chunk_size, total)
        chunk_text = encoding.decode(tokens[start:end]).strip()
        if chunk_text:
            chunks.append(chunk_text)
        if end == total:
            break

    # Merge trailing short chunk (< 50% of chunk_size) into previous
    if len(chunks) >= 2:
        last_len = len(encoding.encode(chunks[-1]))
        if last_len < chunk_size * 0.5:
            tail_start = (len(chunks) - 2) * step
            chunks[-2] = encoding.decode(tokens[tail_start:]).strip()
            chunks.pop()

    return chunks

In [7]:
def chunk_corpus(
    documents: List[Dict[str, Any]],
    chunk_size: int = 300,
    overlap_pct: float = 0.0,
) -> List[Dict[str, Any]]:
    """Chunk all documents and build output records."""
    all_chunks = []

    for doc in documents:
        doc_id = doc.get("doc_id", "unknown")
        text = doc.get("text", "")
        if not text.strip():
            continue

        chunks = fixed_length_token_chunk(text, chunk_size=chunk_size, overlap_pct=overlap_pct)

        for idx, chunk_text in enumerate(chunks):
            record = {
                "chunk_id": f"{doc_id}_chunk_{idx}",
                "text": chunk_text,
                "embedding": [],
                "metadata": {
                    "doc_id": doc_id,
                    "source": doc.get("source", ""),
                    "title": doc.get("title", ""),
                },
            }
            all_chunks.append(record)

    return all_chunks

In [8]:
def save_chunks(chunks: List[Dict], path: str):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for c in chunks:
            f.write(json.dumps(c, ensure_ascii=False) + "\n")
    size_mb = os.path.getsize(path) / (1024 * 1024)
    print(f"Saved {len(chunks)} chunks to {path} ({size_mb:.2f} MB)")


def print_stats(chunks: List[Dict], label: str):
    token_counts = [len(ENCODING.encode(c["text"])) for c in chunks]
    source_counts = Counter(c["metadata"]["source"] for c in chunks)
    print(f"\n--- {label} ---")
    print(f"Total chunks : {len(chunks)}")
    print(f"Tokens  avg={sum(token_counts)/max(len(token_counts),1):.0f}  "
          f"min={min(token_counts, default=0)}  max={max(token_counts, default=0)}")
    for src, cnt in source_counts.most_common():
        print(f"  {src}: {cnt}")

## Run 1: No Overlap (0%)

In [9]:
chunks_no_overlap = chunk_corpus(documents, chunk_size=CHUNK_SIZE, overlap_pct=0.0)
print_stats(chunks_no_overlap, "Fixed 300 tokens, 0% overlap")
save_chunks(chunks_no_overlap, "data/chunks_fixed300_overlap0.jsonl")


--- Fixed 300 tokens, 0% overlap ---
Total chunks : 1838
Tokens  avg=298  min=109  max=447
  Wikipedia: 1672
  Wikibooks: 125
  Around the World in 80 Cuisines Blog: 41
Saved 1838 chunks to data/chunks_fixed300_overlap0.jsonl (2.51 MB)


## Run 2: 20% Overlap

In [10]:
chunks_with_overlap = chunk_corpus(documents, chunk_size=CHUNK_SIZE, overlap_pct=0.2)
print_stats(chunks_with_overlap, "Fixed 300 tokens, 20% overlap")
save_chunks(chunks_with_overlap, "data/chunks_fixed300_overlap20pct.jsonl")


--- Fixed 300 tokens, 20% overlap ---
Total chunks : 2253
Tokens  avg=294  min=109  max=389
  Wikipedia: 2065
  Wikibooks: 140
  Around the World in 80 Cuisines Blog: 48
Saved 2253 chunks to data/chunks_fixed300_overlap20pct.jsonl (3.05 MB)


In [11]:
print(f"{'':30s} {'No Overlap':>12s} {'20% Overlap':>12s}")
print(f"{'Total chunks':30s} {len(chunks_no_overlap):>12d} {len(chunks_with_overlap):>12d}")

t0 = [len(ENCODING.encode(c["text"])) for c in chunks_no_overlap]
t1 = [len(ENCODING.encode(c["text"])) for c in chunks_with_overlap]
print(f"{'Avg tokens/chunk':30s} {sum(t0)/max(len(t0),1):>12.0f} {sum(t1)/max(len(t1),1):>12.0f}")
print(f"{'Total tokens (with repeats)':30s} {sum(t0):>12d} {sum(t1):>12d}")

                                 No Overlap  20% Overlap
Total chunks                           1838         2253
Avg tokens/chunk                        298          294
Total tokens (with repeats)          548111       663504


In [12]:
# Verify output format
for path in ["data/chunks_fixed300_overlap0.jsonl", "data/chunks_fixed300_overlap20pct.jsonl"]:
    with open(path, "r", encoding="utf-8") as f:
        sample = json.loads(f.readline())
    print(f"\n[{os.path.basename(path)}]")
    print(json.dumps(sample, ensure_ascii=False, indent=2))


[chunks_fixed300_overlap0.jsonl]
{
  "chunk_id": "wiki_0001_chunk_0",
  "text": "East Asian cuisine has evolved with common usage of oils, fats and sauces in the preparation of dishes.\n\nChinese cuisine – Traditional Chinese cuisines include Anhui, Cantonese, Fujian, Hunan, Jiangsu, Shandong, Sichuan, and Zhejiang, all of which are defined and termed per the respective regions within China where they developed. A number of different styles contribute to Chinese cuisine, but perhaps the best known and most influential are the Sichuan, Shandong, Jiangsu and Guangdong cuisines. These styles are distinctive from one another due to factors such as available resources, climate, geography, history, cooking techniques and lifestyle. Many Chinese traditional regional cuisines rely on basic methods of food preservation such as drying, salting, pickling and fermentation.\nChinese cuisine originated in what is known as the Eight Great Traditions, though it can be generalized into northern styles